# MS-BART single-spectrum pilot

This notebook tests whether the released MS-BART MassSpecGym checkpoint can generate a molecular SELFIES string from one processed test-set fingerprint. The input is the TSV's `fps` value, which represents MIST-derived spectral information.

The initial run uses one deterministic prediction without reference-formula reranking. The reference SELFIES is used only after generation to evaluate molecular validity and exact structure agreement. This is a setup and feasibility pilot, not yet a matched comparison with the earlier OpenAI prompt experiments.

In [1]:
import sys
import torch

print(sys.executable)
print("CUDA available:", torch.cuda.is_available())

C:\Users\sfnor\miniconda3\envs\ms_bart\python.exe
CUDA available: True


In [2]:
import csv
from pathlib import Path

project_dir = Path.cwd().parent
test_path = project_dir / "ms_bart_pilot" / "data" / "MassSpecGym-MIST_fps_selfies_threshold_0.11.tsv"

with test_path.open(newline="", encoding="utf-8") as file:
    test_row = next(csv.DictReader(file, delimiter="\t"))

print("File found:", test_path.is_file())
print("Identifier:", test_row["identifier"])
print("Fingerprint preview:", test_row["fps"][:100])

File found: True
Identifier: MassSpecGymID0000201
Fingerprint preview: <fp0042><fp0249><fp0389><fp0547><fp0656><fp0695><fp0807><fp0841><fp0875><fp0928><fp1066><fp1088><fp1


In [3]:
import torch
from transformers import BartTokenizer, BartForConditionalGeneration

model_dir = project_dir / "ms_bart_pilot" / "model-weights"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BartTokenizer.from_pretrained(model_dir, local_files_only=True)
model = BartForConditionalGeneration.from_pretrained(
    model_dir, local_files_only=True
).to(device)
model.eval()

print("Model device:", next(model.parameters()).device)

C:\Users\sfnor\miniconda3\envs\ms_bart\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model device: cuda:0


In [4]:
inputs = tokenizer(
    test_row["fps"],
    return_tensors="pt",
    add_special_tokens=False,
)
inputs = {name: tensor.to(device) for name, tensor in inputs.items()}

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        num_beams=1,
    )

predicted_selfies = tokenizer.decode(
    output_ids[0], skip_special_tokens=True
).replace(" ", "")

print("Identifier:", test_row["identifier"])
print("Predicted SELFIES:", predicted_selfies)

Identifier: MassSpecGymID0000201
Predicted SELFIES: [C][O][C][=Branch1][C][=O][C][=C][O][C@@H1][Branch1][C][C][C@H1][C][N][C][C][C@][Branch1][S][C][=Branch1][C][=O][N][C][=C][C][=C][C][=C][Ring1][=Branch1][Ring1][#Branch2][C@@H1][Ring1][=C][C][C@H1][Ring2][Ring1][#Branch1][Ring2][Ring1][C]


In [5]:
from selfies import decoder
from rdkit import Chem

def canonical_smiles(selfies_string):
    try:
        smiles = decoder(selfies_string)
        molecule = Chem.MolFromSmiles(smiles)
        if molecule is None:
            return None
        return Chem.MolToSmiles(molecule, isomericSmiles=True)
    except Exception:
        return None

predicted_smiles = canonical_smiles(predicted_selfies)
reference_smiles = canonical_smiles(test_row["selfies"])

print("Predicted molecule valid:", predicted_smiles is not None)
print("Reference molecule valid:", reference_smiles is not None)
print("Predicted SMILES:", predicted_smiles)
print("Reference SMILES:", reference_smiles)
print("Exact structure match:", predicted_smiles == reference_smiles)

Predicted molecule valid: True
Reference molecule valid: True
Predicted SMILES: COC(=O)C1=CO[C@@H](C)[C@H]2CN3CC[C@]4(C(=O)Nc5ccccc54)[C@@H]3C[C@H]12
Reference SMILES: CC(C)[C@H]1OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@H](C(C)C)OC(=O)[C@H](Cc2ccccc2)N(C)C(=O)[C@@H](C(C)C)OC(=O)[C@H](Cc2ccccc2)N(C)C1=O
Exact structure match: False


### First-row pilot result

For MassSpecGymID00000201, the released MS-BART checkpoint generated a SELFIES string that decoded to a valid RDKit molecule. Its canonical structure did not exactly match the test row's reference structure.

This confirms that the local checkpoint, tokenizer, processed test input, GPU inference, and structure-validation steps work together. One test row does not establish model accuracy. This greedy, single-prediction run also differs from the authors' multi-candidate evaluation and is not a direct comparison with my earlier OpenAI prompt experiments.

Next, I will test a ten-candidate retrieval setup on held-out rows, comparing top-1 selection with the 10% random-choice baseline.